# 🕸️ Mini GraphRAG — Colab 一鍵跑(自己手刻)

> **目標**:用 ~250 行 Python 自己手刻一個 GraphRAG,而不依賴特定庫,讓你真的懂每一步
>
> **架構**:文件 → LLM 抽 entity+relation → networkx graph → 社群偵測 → 摘要 → local/global retrieval
>
> **環境**:Colab CPU(不需 GPU,全 API)、需 OpenAI key
>
> **預期時間**:~10-15 分鐘(視 LLM 速度)
>
> **預期成本**:單次 demo run 約 $0.10-0.30(GPT-4o-mini)

## 為什麼自己手刻而非用 LightRAG / Microsoft GraphRAG

1. **教育價值**:每一步中間結果都看得到(entity 表、graph 邊、社群、摘要)— 對應 deep-dive 中的概念
2. **跑得通**:不依賴 LightRAG / Microsoft GraphRAG 特定 lib 版本(這些 0.x 版本 API 還會大改)
3. **可改造**:每個 step 都可換組件(LLM、graph 演算法、embedding)

**對應 deep-dive**:[`../GraphRAG_hands_on.md`](../GraphRAG_hands_on.md)

## phantom-mesh 寫由

在 phantom-mesh 中,GraphRAG 是處理「多跳查詢」與「主題綜合」的必備元件:
- **incremental indexing**:新文件進來只 re-process 受影響的社群,不重建整個 graph
- **per-tenant graph isolation**:每個 user 一個獨立 graph namespace
- **cost-aware**:indexing 是貴的,要在 freshness 與 cost 間取平衡

---

## 0️⃣ 環境準備

需要 OpenAI API key(或可改 Claude / Gemini,見最後一節)。

In [ ]:
import os, getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
print('✅ API key 已設定')

In [ ]:
%%capture
!pip install -U "openai>=1.50" "networkx>=3.3" "python-louvain>=0.16" "matplotlib" "numpy" "pydantic>=2" "rich"

In [ ]:
import json, re
from collections import defaultdict
from typing import List
import networkx as nx
import community as community_louvain  # python-louvain
import numpy as np
import matplotlib.pyplot as plt
from openai import OpenAI
from pydantic import BaseModel, Field
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

client = OpenAI()
console = Console()
print('✅ Imports OK')

## 1️⃣ 準備文件

用 5 段技術文字當 demo corpus(內建,不依賴外部資料源)。主題環繞 **Attention 機制演進** — 看 GraphRAG 能不能畫出技術演進脈絡。

想換成你自己的文件,把 `DOCUMENTS` 替換即可。

In [ ]:
DOCUMENTS = [
    """Bahdanau Attention 由 Dzmitry Bahdanau 等人在 2014 年論文 'Neural Machine Translation by Jointly Learning to Align and Translate' 提出。
它是 attention 機制在 NMT 的首個應用,讓 decoder 在每個 step 對 encoder 的所有 hidden state 計算 alignment score。
Bahdanau Attention 用 MLP 算 score,稱為 additive attention。後來 Luong attention(2015)改用 dot product,成為主流。
Bahdanau 的方法影響了後續所有 seq2seq 模型,包括 Google Neural Machine Translation 系統。""",

    """Self-Attention 是 Vaswani 等人 2017 年 'Attention Is All You Need' 論文的核心。
與 Bahdanau Attention 不同,Self-Attention 把同一序列中每個 token 都當成 query、key、value,計算彼此關係。
Scaled Dot-Product Attention 公式為 softmax(QK^T/sqrt(d_k))V。Multi-Head Attention 把 attention 切成多個 head 平行運算。
Self-Attention 是 Transformer 的核心,Transformer 取代 RNN/LSTM 成為 NLP 主流架構。
BERT(2018, Google)、GPT(2018, OpenAI)、T5(2019, Google)都基於 Transformer 與 Self-Attention。""",

    """Flash Attention 由 Tri Dao 等人 2022 年論文提出,解決 Self-Attention 在長序列時的 O(N²) memory bottleneck。
傳統 attention 把整個 N×N attention matrix 寫進 HBM(GPU 高頻寬記憶體),Flash Attention 用 tiling 把計算保留在 SRAM。
Flash Attention 2(2023)進一步優化 forward 與 backward。Flash Attention 3(2024)專為 Hopper GPU 設計,使用 WGMMA 與 async pipeline。
Tri Dao 是 Princeton 教授,也是 Together AI 共同創辦人。Flash Attention 已整合進 PyTorch、vLLM、SGLang、TensorRT-LLM。""",

    """Sparse Attention 透過限制 attention 連接的稀疏模式來降低 O(N²) 複雜度。
Longformer(AI2, 2020)用 sliding window + global tokens;BigBird(Google, 2020)用 random + window + global;Sparse Transformer(OpenAI, 2019)用 strided patterns。
2024-2025 重要新進展:DeepSeek 的 Native Sparse Attention(NSA)用 hierarchical compression + selective retention + sliding window,
硬體對齊,FlashAttention-2 forward 9× / backward 6× 加速。Moonshot 的 MoBA(Mixture of Block Attention)把 MoE 思想套到 attention block routing。
這些技術讓 LLM 能擴展到 100K-10M context length。""",

    """RoPE(Rotary Position Embedding,Jianlin Su 2021)是長 context 的另一條關鍵技術線。
RoPE 用旋轉矩陣編碼位置,具有相對位置不變性。NTK-aware scaling、Position Interpolation、YaRN 都是 RoPE 的擴展。
Llama 系列(Meta)、Qwen(Alibaba)、GLM(Tsinghua)、Mistral 都採用 RoPE。
2025 年 Llama 4 Scout 引入 iRoPE(interleaved RoPE),交錯 RoPE 與 no-PE 層,從 256K 訓練外推到 10M context。
LongRoPE(Microsoft)用 evolutionary search 找出 per-dim factor,推到 2M+ context。""",
]

for i, doc in enumerate(DOCUMENTS):
    print(f'--- Doc {i+1} ({len(doc)} chars) ---')
    print(doc[:200] + '...\n')

## 2️⃣ Step 1 — 抽 Entity + Relation

用 GPT-4o-mini + structured output 從每個 doc 抽出實體與三元組關係。這是 GraphRAG 的「indexing」階段最關鍵的步驟。

**phantom-mesh 對應**:這步用「LLM-as-extractor」,可換成 NER 模型加速;production 要 batch + cache。

In [ ]:
class Entity(BaseModel):
    name: str = Field(description='實體名稱,英文或專有名詞保留原文')
    type: str = Field(description='類型:Person / Method / Model / Organization / Year / Concept / Other')
    description: str = Field(description='10-30 字描述')

class Relation(BaseModel):
    source: str = Field(description='源實體 name')
    target: str = Field(description='目標實體 name')
    label: str = Field(description='關係簡述,如 "proposed_by", "based_on", "used_in", "extends"')

class Extraction(BaseModel):
    entities: List[Entity]
    relations: List[Relation]

EXTRACT_PROMPT = """從以下技術文字中抽出實體(人、方法、模型、組織、年份、概念)與關係。
**規則**:
1. entity name 要 canonical(同一個東西用同一名,例如 'Flash Attention' 不要混用 'FlashAttention'/'FA')
2. relation 用簡短英文 label
3. 每篇至少 5 個 entity + 5 個 relation

文字:
{doc}
"""

all_entities = {}  # name -> Entity
all_relations = []

for i, doc in enumerate(DOCUMENTS):
    print(f'\n處理 Doc {i+1}...')
    completion = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': EXTRACT_PROMPT.format(doc=doc)}],
        response_format=Extraction,
    )
    ext = completion.choices[0].message.parsed
    for e in ext.entities:
        if e.name not in all_entities:
            all_entities[e.name] = e
        else:
            # 合併描述
            if len(e.description) > len(all_entities[e.name].description):
                all_entities[e.name] = e
    all_relations.extend(ext.relations)
    print(f'  抽到 {len(ext.entities)} entities, {len(ext.relations)} relations')

print(f'\n✅ 全部完成,共 {len(all_entities)} unique entities, {len(all_relations)} relations')

看一下抽出的 entity 表:

In [ ]:
table = Table(title='Entities')
table.add_column('Name', style='cyan')
table.add_column('Type', style='yellow')
table.add_column('Description')
for e in list(all_entities.values())[:15]:
    table.add_row(e.name, e.type, e.description[:50] + ('...' if len(e.description) > 50 else ''))
console.print(table)
print(f'... 共 {len(all_entities)} entities')

## 3️⃣ Step 2 — 構建 networkx Graph

把 entities 變 node,relations 變 edge。注意 relation 的 source/target 可能是 LLM 寫的變形(大小寫、空格不同),要做 entity resolution。

In [ ]:
def normalize(name: str) -> str:
    return name.strip().lower()

# 建 name → canonical mapping
canonical = {normalize(e.name): e.name for e in all_entities.values()}

G = nx.Graph()
for e in all_entities.values():
    G.add_node(e.name, type=e.type, description=e.description)

for r in all_relations:
    s = canonical.get(normalize(r.source))
    t = canonical.get(normalize(r.target))
    if s and t and s != t:
        if G.has_edge(s, t):
            G[s][t]['labels'].add(r.label)
        else:
            G.add_edge(s, t, labels={r.label})

print(f'✅ Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'   平均 degree: {sum(dict(G.degree()).values())/G.number_of_nodes():.2f}')

## 4️⃣ Step 3 — 可視化 Graph

In [ ]:
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=1.5, seed=42)
# 依 type 上色
color_map = {'Person': 'lightcoral', 'Method': 'lightblue', 'Model': 'lightgreen',
             'Organization': 'orange', 'Year': 'lightyellow', 'Concept': 'lavender', 'Other': 'lightgray'}
node_colors = [color_map.get(G.nodes[n].get('type', 'Other'), 'lightgray') for n in G.nodes()]
nx.draw(G, pos, node_color=node_colors, node_size=800, with_labels=True,
        font_size=8, font_weight='bold', edge_color='gray', alpha=0.7)
plt.title(f'Mini GraphRAG — Attention 機制演進({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)', fontsize=14)
plt.tight_layout()
plt.show()

## 5️⃣ Step 4 — 社群偵測(Louvain)

用 Louvain 演算法找出 graph 中的「社群」— 每個社群是一群緊密相關的 entity。
Microsoft GraphRAG 用 Leiden(Louvain 改進版),核心想法一樣。

In [ ]:
partition = community_louvain.best_partition(G, random_state=42)
communities = defaultdict(list)
for node, cid in partition.items():
    communities[cid].append(node)

console.print(Panel(f'發現 {len(communities)} 個社群', border_style='cyan'))
for cid, members in sorted(communities.items()):
    console.print(f'\n[bold cyan]Community {cid}[/bold cyan] ({len(members)} 成員):')
    console.print('  ' + ', '.join(members[:12]) + ('...' if len(members) > 12 else ''))

## 6️⃣ Step 5 — 社群摘要

對每個社群,讓 LLM 看完所有成員與彼此關係後寫一段摘要。這份摘要是 **global search** 的核心 — 用戶問「整個領域怎樣」時,我們檢索的是這些摘要而不是原文。

In [ ]:
SUMMARY_PROMPT = """以下是一個技術社群的實體與關係:

{community_text}

請用繁體中文寫一段 80-150 字的社群摘要,涵蓋:
1. 這個社群的主題是什麼
2. 核心 entity / 關鍵 method 是什麼
3. 內部最重要的關係或演進線
"""

community_summaries = {}
for cid, members in communities.items():
    if len(members) < 2:
        community_summaries[cid] = f'(社群只有 {members[0]},不需摘要)'
        continue
    text_parts = []
    for m in members:
        text_parts.append(f'- {m}({G.nodes[m].get("type")}):{G.nodes[m].get("description")}')
    text_parts.append('\n關係:')
    for u, v, d in G.edges(members, data=True):
        if u in members and v in members:
            text_parts.append(f'  {u} —[{",".join(d.get("labels", []))}]— {v}')
    community_text = '\n'.join(text_parts)
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': SUMMARY_PROMPT.format(community_text=community_text)}],
    )
    community_summaries[cid] = resp.choices[0].message.content
    console.print(Panel(
        community_summaries[cid],
        title=f'Community {cid} ({len(members)} 成員)',
        border_style='cyan',
    ))

## 7️⃣ Local Search — 具體事實查詢

Local search 處理「具體實體相關」的問題。流程:
1. LLM 從 query 抽 entity
2. 找 graph 中相符 / 鄰近的 entity
3. 把這些 entity + 它們的鄰居 + 相關 edges 餵 LLM 答題

In [ ]:
def local_search(query: str, k_hop: int = 1) -> str:
    # 從 query 找最相關的 entity(naive: 字串匹配;production 用 embedding)
    matched = [n for n in G.nodes() if n.lower() in query.lower() or any(part in n.lower() for part in query.lower().split())]
    if not matched:
        return '(找不到相關 entity)'
    # 收集 k-hop 鄰居
    subgraph_nodes = set(matched)
    for n in matched:
        subgraph_nodes.update(G.neighbors(n))
    context_parts = ['=== 相關 entity 與描述 ===']
    for n in subgraph_nodes:
        context_parts.append(f'- {n} ({G.nodes[n]["type"]}): {G.nodes[n]["description"]}')
    context_parts.append('\n=== 相關 relations ===')
    for u, v, d in G.subgraph(subgraph_nodes).edges(data=True):
        context_parts.append(f'  {u} —[{",".join(d.get("labels", []))}]— {v}')
    context = '\n'.join(context_parts)
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': '你是技術助手。根據提供的 graph context 回答問題,引用具體 entity 名。'},
            {'role': 'user', 'content': f'問題:{query}\n\nGraph context:\n{context}'},
        ],
    )
    return resp.choices[0].message.content

Q1 = 'Flash Attention 是誰提出的?他還做了什麼?'
console.print(Panel(local_search(Q1), title=f'Local Search:{Q1}', border_style='green'))

## 8️⃣ Global Search — 主題綜合查詢

Global search 處理「整體趨勢、主題綜合」的問題。流程:
1. 把所有 community summaries 餵 LLM
2. LLM 依問題綜合

Microsoft GraphRAG 的「global search」 mode 就是這個概念。

In [ ]:
def global_search(query: str) -> str:
    summaries_text = '\n\n'.join(
        f'### Community {cid}\n{s}' for cid, s in community_summaries.items()
    )
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': '你是技術趨勢分析師。基於以下社群摘要綜合回答問題,提供 high-level 觀點。'},
            {'role': 'user', 'content': f'問題:{query}\n\n社群摘要:\n{summaries_text}'},
        ],
    )
    return resp.choices[0].message.content

Q2 = '從整體看,attention 機制這幾年的演進主軸是什麼?有哪些主要技術線?'
console.print(Panel(global_search(Q2), title=f'Global Search:{Q2}', border_style='magenta'))

## 9️⃣ Naive RAG Baseline — 對比

用簡單的 keyword 檢索原文當 baseline,看 GraphRAG 跟 vanilla RAG 差在哪。

In [ ]:
def naive_rag(query: str) -> str:
    # 簡單 keyword 匹配 top-2 doc
    scores = []
    for doc in DOCUMENTS:
        score = sum(1 for w in query.split() if w in doc)
        scores.append(score)
    top = sorted(range(len(DOCUMENTS)), key=lambda i: -scores[i])[:2]
    context = '\n\n'.join(DOCUMENTS[i] for i in top)
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': '根據提供的文件回答問題。'},
            {'role': 'user', 'content': f'問題:{query}\n\n文件:\n{context}'},
        ],
    )
    return resp.choices[0].message.content

console.print(Panel(naive_rag(Q2), title=f'Naive RAG:{Q2}', border_style='yellow'))

## 1️⃣0️⃣ 三模式對比

理論上對「主題綜合」類問題,Global Search 應該勝;對「具體事實」類問題,Local 勝;Naive RAG 處於中間。

In [ ]:
Q3 = '長 context 技術中,Flash Attention 與 Sparse Attention 的差別?'
console.print('\n[bold]Q3:[/bold]', Q3)
console.print(Panel(local_search(Q3), title='Local Search', border_style='green'))
console.print(Panel(global_search(Q3), title='Global Search', border_style='magenta'))
console.print(Panel(naive_rag(Q3), title='Naive RAG', border_style='yellow'))

## 1️⃣1️⃣ phantom-mesh 真實工程考量

### 11.1 Incremental Indexing
- 新文件進來,只 re-process 受影響的社群,不重建整個 graph
- 用 graph diff 偵測哪些 community 的成員變了 → 只重 summary 那些

### 11.2 Entity Resolution
- 本 demo 用 lowercase normalize,生產要用 embedding 相似度 + LLM verify
- 例如 'FlashAttention'、'Flash Attention'、'FA' 都要 merge

### 11.3 Cost Control
- 抽 entity 是 1 LLM call / chunk;摘要是 1 call / community
- 預算 cap:per-document indexing cost < $0.10
- 對應 [Case_02 §5.5 Cost Tracker](../../../9.面試準備與職業發展/2.系統設計案例/Case_02_LLM_Gateway_API_Platform.md)

### 11.4 Per-tenant Graph Isolation
- 每個 user / company 一個獨立 graph namespace
- Neo4j label-based RBAC 或 Postgres + pgvector schema 隔離

### 11.5 Freshness vs Cost
- 全量 reindex:乾淨但貴
- Incremental:快但可能漂
- 折衷:每天小幅 incremental + 每週夜間全量 reindex

### 11.6 Hybrid Routing
- Query 分類:具體事實 → Local;主題綜合 → Global;事實+綜合混合 → 兩者都跑
- Router 可以用小模型分類(節省成本)

---

## 🔬 擴展練習

1. **換用 Microsoft GraphRAG 官方套件**:`pip install graphrag` 跑同樣資料,比較結果與成本
2. **換用 LightRAG**:`pip install lightrag-hku`,看 dual-level retrieval 的差別
3. **加 vector retrieval**:對每個 chunk 跑 embedding,Local Search 結合 graph + vector
4. **接 Neo4j**:把 networkx graph 寫進 Neo4j,用 Cypher 查
5. **加 fact-checker**:答案產生後,回查 graph 中是否有支持證據
6. **加 Causal layer**:對 entity 抽 causal relation(連到 [`../../17.Causal_ML/`](../../../17.Causal_ML/README.md))
7. **多模態**:加圖像 entity(用 [ColPali](../ColPali_Late_Chunking_Contextual_Retrieval.md))
8. **整合進 multi-agent**:把 GraphRAG 當 retrieval tool,被 [LangGraph supervisor](../../3.Agent/notebooks/Colab_LangGraph_Multi_Agent_Research_Demo.ipynb) 呼叫

---

## 📚 References

- [Microsoft GraphRAG 官方](https://github.com/microsoft/graphrag)
- [LightRAG (HKU) GitHub](https://github.com/HKUDS/LightRAG)
- [HippoRAG 2 paper](https://arxiv.org/abs/2502.14802)
- 本 repo:[`../GraphRAG_hands_on.md`](../GraphRAG_hands_on.md)、[`../ColPali_Late_Chunking_Contextual_Retrieval.md`](../ColPali_Late_Chunking_Contextual_Retrieval.md)、[`../../../18.GNN_Graph_Learning/`](../../../18.GNN_Graph_Learning/README.md)

---

**Last updated**: 2026-05-16  
**Tested on**: Colab CPU,Python 3.10,openai 1.50,networkx 3.3,python-louvain 0.16